# CASMI 2026 — Phase 3: Learned Fingerprint Prediction
**Phase 2 library search + MLP fingerprint predictor for novel structures.**

Validation is structure-disjoint: np-examples structures excluded from training AND library.

In [ ]:
import numpy as np, pandas as pd, pyarrow.parquet as pq
import os, time, glob
from collections import defaultdict
from hashlib import md5
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print("=== /kaggle/input ===")
for root, dirs, files in os.walk("/kaggle/input"):
    lvl = root.replace("/kaggle/input", "").count(os.sep)
    if lvl > 2: continue
    print(" " * lvl + os.path.basename(root) + "/")

In [ ]:
t0 = time.time()
tpath = glob.glob("/kaggle/input/**/train.parquet", recursive=True)[0]
tepath = glob.glob("/kaggle/input/**/test.parquet", recursive=True)[0]
tr = pq.read_table(tpath).to_pandas()
te_df = pq.read_table(tepath).to_pandas()
print(f"train {tr.shape} test {te_df.shape} ({time.time()-t0:.0f}s)")
print("train:", list(tr.columns))
print("test:", list(te_df.columns))

In [ ]:
# ============ Core functions ============
ADDUCTS = {"[M+H]+": 1.00727647, "[M+Na]+": 22.98921870, "[M+K]+": 38.96315853,
    "[M+NH4]+": 18.03382307, "[M-H]-": -1.00727647, "[M+Cl]-": 34.96940248,
    "[M+CH2O2-H]-": 44.99820131, "[M+CH3OH+H]+": 33.03348979, "[M+ACN+H]+": 42.03382307}
def neutral_mass(prec, ad):
    if ad in ADDUCTS: return prec - ADDUCTS[ad]
    if ad.startswith("[2M") and ad.endswith("]+"):
        b = {"+H": 1.00727647, "+Na": 22.98921870, "+K": 38.96315853, "+NH4": 18.03382307}.get(ad[3:-2])
        if b is not None: return (prec - b) / 2.0
    return np.nan

def preprocess_peaks(mzs, ints, top_n=50):
    mzs, ints = np.asarray(mzs, float), np.asarray(ints, float)
    k = ints > 0; mzs, ints = mzs[k], ints[k]
    if len(mzs) == 0: return mzs, ints
    o = np.argsort(-ints)[:top_n]; mzs, ints = mzs[o], ints[o]
    ints = np.sqrt(ints); n = np.linalg.norm(ints)
    if n > 0: ints /= n
    o2 = np.argsort(mzs); return mzs[o2], ints[o2]

def entropy_sim(mz1, i1, mz2, i2, tol=0.01):
    if len(mz1) == 0 or len(mz2) == 0: return 0.0
    am = np.concatenate([mz1, mz2]); ai = np.concatenate([i1, -i2])
    o = np.argsort(am); am, ai = am[o], ai[o]
    p1, p2 = [], []
    cm, cp1, cp2 = am[0], 0.0, 0.0
    for m, v in zip(am, ai):
        if m - cm <= tol:
            if v > 0: cp1 += v
            else: cp2 -= v
        else:
            p1.append(cp1); p2.append(cp2); cm, cp1, cp2 = m, max(v, 0), max(-v, 0)
    p1.append(cp1); p2.append(cp2)
    p1, p2 = np.array(p1), np.array(p2); s1, s2 = p1.sum(), p2.sum()
    if s1 == 0 or s2 == 0: return 0.0
    p1, p2 = p1 / s1, p2 / s2; m = 0.5 * (p1 + p2)
    def H(p):
        p = p[p > 0]; return -np.sum(p * np.log(p))
    js = H(m) - 0.5 * H(p1) - 0.5 * H(p2)
    return float(1.0 - js / np.log(2))

In [ ]:
# ============ Build library index ============
t0 = time.time()
tr_prec = tr["precursor_mz"].values; tr_adduct = tr["adduct"].values
sm_col = "smiles" if "smiles" in tr.columns else "Smiles"
tr_smiles = tr[sm_col].values
key_col = "inchikey" if "inchikey" in tr.columns else None
if key_col:
    tr_key = np.array([str(k)[:14] for k in tr[key_col].values])
else:
    tr_key = np.array([md5(str(s).encode()).hexdigest()[:14] for s in tr_smiles])
nm = np.array([neutral_mass(p, a) for p, a in zip(tr_prec, tr_adduct)])
valid = ~np.isnan(nm)
print(f"valid adducts: {valid.sum()}/{len(tr)}")
mz_col = "mzs" if "mzs" in tr.columns else "mz"
in_col = "intensities" if "intensities" in tr.columns else "intensity"
kept = np.nonzero(valid)[0]
lmz, lint, lneu, lkey = [], [], [], []
for i in kept:
    mz, it = preprocess_peaks(np.asarray(tr[mz_col].values[i]), np.asarray(tr[in_col].values[i]))
    if len(mz) < 3: continue
    lmz.append(mz); lint.append(it); lneu.append(nm[i]); lkey.append(tr_key[i])
lmz = np.array(lmz, dtype=object); lint = np.array(lint, dtype=object)
lneu = np.array(lneu); lkey = np.array(lkey)
o = np.argsort(lneu); lmz, lint, lneu, lkey = lmz[o], lint[o], lneu[o], lkey[o]
key_to_smiles = {}
for k, s in zip(lkey, tr_smiles[valid][:len(lkey)]):
    if k not in key_to_smiles: key_to_smiles[k] = str(s)
print(f"index: {len(lkey)} spectra, {len(key_to_smiles)} structures ({time.time()-t0:.0f}s)")

In [ ]:
# ============ Fingerprints ============
t0 = time.time()
keys = list(key_to_smiles.keys()); NK = len(keys)
fp_packed = np.zeros((NK, 256), dtype=np.uint8)
for j, k in enumerate(keys):
    s = key_to_smiles[k]
    for kk in (3, 4):
        for i in range(len(s) - kk + 1):
            h = int(md5(s[i:i+kk].encode()).hexdigest(), 16) % 2048
            fp_packed[j, h // 8] |= (1 << (h % 8))
key_to_idx = {k: j for j, k in enumerate(keys)}
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint16)
fp_bitsum = POP[fp_packed].sum(axis=1)
print(f"fingerprints: {NK} ({time.time()-t0:.0f}s)")

In [ ]:
# ============ MLP definition ============
N_BINS = 2000; MZ_MIN = 50.0; MZ_MAX = 1500.0
def bin_spectrum(mzs, ints):
    b = np.zeros(N_BINS, dtype=np.float32)
    if len(mzs) == 0: return b
    idx = ((mzs - MZ_MIN) / (MZ_MAX - MZ_MIN) * N_BINS).astype(int)
    v = (idx >= 0) & (idx < N_BINS); idx = idx[v]
    vals = np.sqrt(ints[v].astype(np.float32))
    for i, x in zip(idx, vals):
        if x > b[i]: b[i] = x
    n = np.linalg.norm(b)
    if n > 0: b /= n
    return b

class FpMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_BINS + 8, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024, 2048))
    def forward(self, x): return self.net(x)
print("MLP defined")

In [ ]:
# ============ Training data (structure-disjoint) ============
t0 = time.time()
np_col = "is_np_example" if "is_np_example" in tr.columns else None
if np_col:
    np_mask_full = tr[np_col].values
else:
    np_mask_full = np.zeros(len(tr), bool)
# Map to library indices: need tr_key for valid rows
np_keys = set(tr_key[valid][np_mask_full[valid]]) if np_col else set()
print(f"held-out structures: {len(np_keys)}")
train_candidates = [i for i in range(len(lkey)) if lkey[i] not in np_keys]
np.random.seed(42); np.random.shuffle(train_candidates)
N_TRAIN = min(200000, len(train_candidates))
train_idx = train_candidates[:N_TRAIN]
print(f"train spectra: {N_TRAIN} ({time.time()-t0:.0f}s)")

In [ ]:
# ============ Dataset ============
class FpDataset(Dataset):
    def __init__(self, indices):
        self.idx = indices
        self._unpack = np.unpackbits(np.arange(256, dtype=np.uint8)[:, None], axis=1)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        binned = bin_spectrum(lmz[j], lint[j])
        prec_norm = np.float32((lneu[j] - 500.0) / 1000.0)
        x = np.concatenate([binned, [prec_norm], np.zeros(7, dtype=np.float32)]).astype(np.float32)
        fp_bits = self._unpack[fp_packed[key_to_idx[lkey[j]]]].astype(np.float32).ravel()
        return torch.from_numpy(x), torch.from_numpy(fp_bits)

train_ds = FpDataset(train_idx)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=2, pin_memory=True)
print(f"loader: {len(train_ds)} samples")

In [ ]:
# ============ Train ============
model = FpMLP().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
crit = nn.BCEWithLogitsLoss()
for ep in range(2):
    model.train(); tot = 0.0; n = 0; t0 = time.time()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += loss.item() * len(xb); n += len(xb)
    print(f"Epoch {ep+1}/2: loss={tot/n:.4f} ({time.time()-t0:.0f}s)", flush=True)
print("training done")

In [ ]:
# ============ Validation (structure-disjoint, FIXED) ============
def library_search(qmz, qi, qneu, exclude_keys=None, top_k=30):
    lo, hi = qneu * 0.99999, qneu * 1.00001
    l = np.searchsorted(lneu, lo); r = np.searchsorted(lneu, hi)
    scored = []
    for j in range(l, r):
        if exclude_keys and lkey[j] in exclude_keys: continue
        s = entropy_sim(qmz, qi, lmz[j], lint[j])
        if s > 0.01: scored.append((s, lkey[j]))
    scored.sort(reverse=True)
    best = {}
    for s, k in scored:
        if k not in best: best[k] = s
    return sorted(best.items(), key=lambda x: -x[1])[:top_k]

def predict_fp(spectra_list):
    model.eval(); preds = []
    with torch.no_grad():
        for mzs, ints, neu in spectra_list:
            binned = bin_spectrum(mzs, ints)
            prec_norm = np.float32((neu - 500.0) / 1000.0)
            x = np.concatenate([binned, [prec_norm], np.zeros(7, dtype=np.float32)]).astype(np.float32)
            xb = torch.from_numpy(x).unsqueeze(0).to(DEVICE)
            preds.append(torch.sigmoid(model(xb)).cpu().numpy()[0])
    return np.mean(preds, axis=0)

def fp_search(query_fp, top_k=25):
    qbits = (query_fp > 0.5).astype(np.uint8)
    qpacked = np.packbits(qbits).reshape(1, 256)
    inter = POP[np.bitwise_and(fp_packed, qpacked)].sum(axis=1)
    union = qbits.sum() + fp_bitsum - inter
    sim = np.divide(inter, union, out=np.zeros(NK), where=union > 0)
    top = np.argpartition(-sim, top_k)[:top_k]
    top = top[np.argsort(-sim[top])]
    return [(keys[i], float(sim[i])) for i in top]

# Build validation molecules
np_by_mol = defaultdict(list)
for j in range(len(lkey)):
    if lkey[j] in np_keys:
        np_by_mol[lkey[j]].append(j)
val_mols = list(np_by_mol.keys())[:200]
print(f"validating on {len(val_mols)} held-out molecules")
rr = 0.0
for vi, true_k in enumerate(val_mols):
    crows = np_by_mol[true_k]
    lib_hits = []
    for cr in crows[:2]:
        lib_hits.extend(library_search(lmz[cr], lint[cr], lneu[cr], exclude_keys=np_keys))
    spectra = [(lmz[cr], lint[cr], lneu[cr]) for cr in crows[:2]]
    qfp = predict_fp(spectra)
    fp_hits = fp_search(qfp)
    K = 60; fused = defaultdict(float); seen = set()
    for k, s in sorted(lib_hits, key=lambda x: -x[1]):
        if k in seen: continue
        seen.add(k); fused[k] += 1.0 / (K + len(seen))
    for rank, (k, s) in enumerate(fp_hits, 1):
        fused[k] += 1.0 / (K + rank)
    ranked = sorted(fused.items(), key=lambda x: -x[1])[:25]
    for rank, (k, s) in enumerate(ranked, 1):
        if k == true_k:
            rr += 1.0 / rank; break
    if (vi + 1) % 50 == 0:
        print(f"  {vi+1} MRR={rr/(vi+1):.4f}", flush=True)
print(f"VALIDATION MRR@25 (Phase 3): {rr/len(val_mols):.4f}")

In [ ]:
# ============ Test prediction ============
mol_col = None
for c in te_df.columns:
    if "mol" in c.lower():
        mol_col = c; break
te_mols = te_df[mol_col].unique() if mol_col else range(len(te_df))
print(f"test molecules: {len(te_mols)}")
mz_col = "mzs" if "mzs" in te_df.columns else "mz"
in_col = "intensities" if "intensities" in te_df.columns else "intensity"

def predict_molecule(spectra):
    lib_hits = []
    fp_spectra = []
    for mzs, ints, prec, ad in spectra:
        nm_ = neutral_mass(prec, ad)
        if np.isnan(nm_): continue
        qmz, qi = preprocess_peaks(mzs, ints)
        lib_hits.extend(library_search(qmz, qi, nm_))
        fp_spectra.append((qmz, qi, nm_))
    fp_hits = fp_search(predict_fp(fp_spectra)) if fp_spectra else []
    K = 60; fused = defaultdict(float); seen = set()
    for k, s in sorted(lib_hits, key=lambda x: -x[1]):
        if k in seen: continue
        seen.add(k); fused[k] += 1.0 / (K + len(seen))
    for rank, (k, s) in enumerate(fp_hits, 1):
        fused[k] += 1.0 / (K + rank)
    ranked = [k for k, _ in sorted(fused.items(), key=lambda x: -x[1])[:25]]
    smiles_list = []
    for k in ranked:
        s = key_to_smiles.get(k)
        if s and s not in smiles_list: smiles_list.append(s)
    while len(smiles_list) < 25:
        smiles_list.append(smiles_list[-1] if smiles_list else "")
    return smiles_list[:25]

t0 = time.time(); rows = []
for mi, mol in enumerate(te_mols):
    sub = te_df[te_df[mol_col] == mol] if mol_col else te_df.iloc[[mol]]
    spectra = []
    for _, row in sub.iterrows():
        spectra.append((np.asarray(row[mz_col]), np.asarray(row[in_col]),
                       row["precursor_mz"], row["adduct"] if "adduct" in te_df.columns else "[M+H]+"))
    rows.append(";".join(predict_molecule(spectra)))
    if (mi + 1) % 50 == 0: print(f"  {mi+1}/{len(te_mols)} ({time.time()-t0:.0f}s)", flush=True)
sub_df = pd.DataFrame({"id": range(len(rows)), "smiles": rows})
sub_df.to_csv("/kaggle/working/submission.csv", index=False)
print(f"done ({time.time()-t0:.0f}s) shape={sub_df.shape}")